# KAN-Former-1D (paper4) — End-to-End Colab Notebook

**KAN-Former-1D**: PyTorch teacher model for 5-class AAMI ECG arrhythmia classification.  
Uses the de Chazal DS1/DS2 inter-patient split (same patients as paper3).  
Produces soft probability labels for knowledge distillation into the paper3 student.

**Runtime:** GPU recommended (`Runtime → Change runtime type → T4 GPU`).  
CPU-only training works but is slow (~4× longer per epoch).

In [ ]:
# ── Cell 1: Detect environment and mount Drive ────────────────────────────────
import sys
import os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if not IN_COLAB:
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

print(f'Running in Colab: {IN_COLAB}')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Create a persistent project directory on Drive
    PROJECT_DIR = Path('/content/drive/MyDrive/fyp/HasanML/paper4')
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Project dir: {PROJECT_DIR}')
else:
    PROJECT_DIR = Path('.').resolve()
    print(f'Local run from: {PROJECT_DIR}')

In [ ]:
# ── Cell 2: Clone or pull the repo (Colab only) ───────────────────────────────
if IN_COLAB:
    REPO_DIR = Path('/content/HasanML')
    if not REPO_DIR.exists():
        # Clone the repo (replace with your actual repo URL)
        import subprocess
        result = subprocess.run(
            ['git', 'clone', 'https://github.com/YOUR_USERNAME/HasanML.git', str(REPO_DIR)],
            capture_output=True, text=True
        )
        print(result.stdout, result.stderr)
    else:
        import subprocess
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], capture_output=True)
        print('Repo updated')

    PAPER4_DIR = REPO_DIR / 'paper4'
    sys.path.insert(0, str(PAPER4_DIR))
    os.chdir(PAPER4_DIR)
    print(f'Working dir: {os.getcwd()}')
else:
    PAPER4_DIR = PROJECT_DIR
    # Already in paper4 dir if running locally
    if (Path.cwd() / 'src').exists():
        pass
    elif (Path.cwd().parent / 'src').exists():
        os.chdir(Path.cwd().parent)
    print(f'Working dir: {os.getcwd()}')

In [ ]:
# ── Cell 3: Install dependencies ──────────────────────────────────────────────
# PyTorch is pre-installed on Colab; install only the extras.
# On local: install PyTorch first (see requirements.txt comments).

if IN_COLAB:
    !pip install -q wfdb scipy scikit-learn lightgbm pyyaml tqdm onnx
else:
    print('Local install: run  pip install -r requirements.txt  if not done')

# Verify PyTorch
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 4: Configure device and mixed precision ──────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MIXED_PRECISION = torch.cuda.is_available()  # auto-enable on GPU

if DEVICE == 'cuda' and MIXED_PRECISION:
    # Enable TF32 for extra speed on Ampere/Ada GPUs (T4, A100, RTX 4000+)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f'GPU training: mixed_precision=True (AMP float16/bfloat16)')
    if torch.cuda.is_bf16_supported():
        AMP_DTYPE = torch.bfloat16
        print('  → using bfloat16 (Ampere+, more stable than float16)')
    else:
        AMP_DTYPE = torch.float16
        print('  → using float16')
else:
    print(f'CPU training: mixed_precision=False')
    AMP_DTYPE = None

print(f'Device: {DEVICE}')

In [ ]:
# ── Cell 5: MIT-BIH data setup ────────────────────────────────────────────────
# wfdb.dl_database downloads records on demand when build_dataset() is called.
# Here we just configure the data directory.

DATA_DIR = Path('data/mitbih')   # relative to paper4/
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Optional: pre-download all records to avoid per-record delays during training
PREDOWNLOAD = False   # set to True to download all 44 records now (~100 MB)

if PREDOWNLOAD:
    import wfdb
    from src.data import DS1, DS2
    all_records = DS1 + DS2
    print(f'Downloading {len(all_records)} MIT-BIH records to {DATA_DIR} ...')
    wfdb.dl_database('mitdb', dl_dir=str(DATA_DIR), records=all_records, annotators=['atr'])
    print('Download complete.')
else:
    print(f'Data dir: {DATA_DIR}  (records will auto-download on first access)')

In [ ]:
# ── Cell 6: Run unit tests ────────────────────────────────────────────────────
# 34 tests covering: split isolation, AAMI mapping, RR causality,
# window centering, KANLinear (output/gradients/basis/sin fit),
# focal loss, metrics CM, soft label format.
# Tests that require the network run on CPU only (fast, ~15 s).

import subprocess
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '-v', '--tb=short'],
    capture_output=False
)
if result.returncode != 0:
    print('\n⚠️  Some tests failed. Check output above before proceeding.')
else:
    print('\n✓  All tests passed.')

In [ ]:
# ── Cell 7: RR-only LightGBM sanity check ────────────────────────────────────
# Verifies RR features are correct: should give ~90-93% acc, Se_S ~60%.
# Fast (~1-2 min). If far off, something is wrong with the RR computation.

import subprocess
result = subprocess.run(
    [sys.executable, '-m', 'src.run_experiment',
     '--rr-baseline',
     '--data-dir', str(DATA_DIR)],
    capture_output=False
)
print('RR baseline complete.' if result.returncode == 0 else 'RR baseline failed.')

In [ ]:
# ── Cell 8: Full training (5-class AAMI, 3 seeds) ────────────────────────────
# Trains 3 seeds on DS1 (train+val), evaluates on DS2.
# GPU: ~20-40 min for 3 seeds (depends on hardware).
# CPU: ~3-4 hours for 3 seeds.

RESULTS_DIR = Path('results/colab_run')

cmd = [
    sys.executable, '-m', 'src.run_experiment',
    '--classes', '5',
    '--seeds', '3',
    '--device', DEVICE,
    '--results-dir', str(RESULTS_DIR),
    '--data-dir', str(DATA_DIR),
]

if MIXED_PRECISION:
    cmd.append('--mixed-precision')

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=False)
print('Training complete.' if result.returncode == 0 else f'Training failed (rc={result.returncode}).')

In [ ]:
# ── Cell 9: Distillation export ───────────────────────────────────────────────
# Loads the trained models, calibrates temperature, exports:
#   results/colab_run/distill/soft_labels_{split}_T{1,2,4}.npy
#   results/colab_run/distill/model_ensemble.onnx

result = subprocess.run(
    [sys.executable, '-m', 'src.run_experiment',
     '--distill',
     '--results-dir', str(RESULTS_DIR),
     '--data-dir', str(DATA_DIR),
     '--device', DEVICE],
    capture_output=False
)

if result.returncode == 0:
    distill_dir = RESULTS_DIR / 'distill'
    if distill_dir.exists():
        for f in sorted(distill_dir.iterdir()):
            size = f.stat().st_size / 1e6
            print(f'  {f.name}  ({size:.2f} MB)')
    print('Distillation export complete.')
else:
    print('Distillation failed.')

In [ ]:
# ── Cell 10: Copy results to Drive (Colab only) ───────────────────────────────
if IN_COLAB:
    import shutil
    dest = PROJECT_DIR / 'results' / 'colab_run'
    dest.parent.mkdir(parents=True, exist_ok=True)
    if RESULTS_DIR.exists():
        shutil.copytree(str(RESULTS_DIR), str(dest), dirs_exist_ok=True)
        print(f'Results saved to Drive: {dest}')
    else:
        print('No results to copy (training may not have completed).')
else:
    print(f'Local run — results are at {RESULTS_DIR.resolve()}')

In [ ]:
# ── Cell 11 (optional): Quick ablation run ────────────────────────────────────
# Uncomment to run all 7 ablations (A1-A6b).
# This takes significantly longer (~7× training time).

# result = subprocess.run(
#     [sys.executable, '-m', 'src.run_experiment',
#      '--ablations',
#      '--seeds', '3',
#      '--device', DEVICE,
#      '--results-dir', str(RESULTS_DIR),
#      '--data-dir', str(DATA_DIR)]
#      + (['--mixed-precision'] if MIXED_PRECISION else []),
#     capture_output=False
# )
# print('Ablations complete.' if result.returncode == 0 else 'Ablations failed.')
print('Cell 11: uncomment to run ablations.')